# SafeVision AI — Cloud PPE Model Training (Google Colab + Google Drive)

This notebook trains a custom **YOLOv8** Personal Protective Equipment (PPE) detection model on Google Colab's free GPU (Tesla T4) and **automatically persists all checkpoints and exported weights to Google Drive**.

### Pipeline Overview:
1. **Mount Google Drive** ➔ Persistent storage (No data loss on disconnect)
2. **GPU Check** ➔ Verify NVIDIA CUDA acceleration
3. **Dependency Setup** ➔ Ultralytics, Roboflow, ONNX
4. **Dataset Download** ➔ Industrial PPE dataset saved directly in Drive
5. **YOLOv8 Fine-Tuning** ➔ 50 Epochs with automatic checkpointing
6. **Validation & Metrics** ➔ Confusion matrix, Precision, Recall, mAP50
7. **ONNX FP16 Quantization** ➔ Edge-optimized export (<25ms inference)
8. **Download Package** ➔ Bundles `best.pt` & `best.onnx` for local machine

In [ ]:
# Cell 1: Mount Google Drive & Create Project Directory
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define dedicated directory in your Google Drive
PROJECT_DIR = '/content/drive/MyDrive/SafeVision_AI'
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"✅ Google Drive successfully mounted!")
print(f"📁 Persistent Project Folder: {PROJECT_DIR}")

In [ ]:
# Cell 2: Verify GPU Hardware & CUDA
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Acceleration Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Assigned GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Install Required Dependencies
!pip install --upgrade ultralytics roboflow onnx onnxslim

In [ ]:
# Cell 4: Download Industrial PPE Dataset directly into Google Drive
import os

DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

if not os.path.exists(DATASET_DIR) or len(os.listdir(DATASET_DIR)) == 0:
    os.makedirs(DATASET_DIR, exist_ok=True)
    print("Downloading industrial PPE dataset to Google Drive...")
    !curl -L "https://universe.roboflow.com/ds/j3Vrhv0VpL?key=demo" > /content/ppe_dataset.zip
    !unzip -q /content/ppe_dataset.zip -d "{DATASET_DIR}"
    !rm /content/ppe_dataset.zip
    print(f"✅ Dataset successfully stored in: {DATASET_DIR}")
else:
    print(f"✅ Dataset already exists at {DATASET_DIR}. Skipping re-download!")

In [ ]:
# Cell 5: Train YOLOv8 with Checkpoints Saved Directly to Google Drive
import os
from ultralytics import YOLO

data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")

# Start with pretrained YOLOv8 nano backbone
model = YOLO('yolov8n.pt')

# Run fine-tuning (All checkpoints persist in Google Drive)
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=os.path.join(PROJECT_DIR, "runs"),
    name="ppe_training_run",
    exist_ok=True,
    save=True,
    val=True
)

print("✅ Training completed! Weights saved to Google Drive:")
print(os.path.join(PROJECT_DIR, "runs", "ppe_training_run", "weights"))

In [ ]:
# Cell 6: Visualize Performance & Evaluation Metrics
import os
from IPython.display import Image, display

run_dir = os.path.join(PROJECT_DIR, "runs", "ppe_training_run")

# Display Confusion Matrix
cm_path = os.path.join(run_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    print("📊 Confusion Matrix:")
    display(Image(cm_path))

# Display mAP and Loss Curves
res_path = os.path.join(run_dir, "results.png")
if os.path.exists(res_path):
    print("📈 Training Results (Loss & mAP Curves):")
    display(Image(res_path))

In [ ]:
# Cell 7: Export to ONNX FP16 for Edge Real-Time Inference
import os
from ultralytics import YOLO

best_weights = os.path.join(PROJECT_DIR, "runs", "ppe_training_run", "weights", "best.pt")
trained_model = YOLO(best_weights)

# Export half-precision (FP16) model with dynamic axes
onnx_export_path = trained_model.export(
    format="onnx",
    half=True,
    dynamic=True
)
print(f"✅ Exported ONNX FP16 Model: {onnx_export_path}")

In [ ]:
# Cell 8: Package Trained Weights into Zip Bundle for Local Download
import os
import shutil
from google.colab import files

export_package_dir = os.path.join(PROJECT_DIR, "export_package")
os.makedirs(export_package_dir, exist_ok=True)

weights_dir = os.path.join(PROJECT_DIR, "runs", "ppe_training_run", "weights")
for f in ["best.pt", "best.onnx"]:
    src = os.path.join(weights_dir, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(export_package_dir, f))

zip_path = os.path.join(PROJECT_DIR, "SafeVision_AI_Weights.zip")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', export_package_dir)

print(f"📦 Package created at: {zip_path}")
print("Triggering automatic browser download...")
files.download(zip_path)